In [18]:
import json
import pandas as pd
from IPython.display import display

with (RELEASE_DIR / "manifest.json").open(
    encoding="utf-8-sig"
) as file:
    manifest_data = json.load(file)

# Giữ ID dạng chuỗi; không biến mã quốc gia "NA" thành giá trị thiếu.
identifier_types = {
    "tournament_id": "string",
    "placement_id": "string",
    "source_tournament_id": "string",
    "source_team_id": "string",
    "source_team_id_raw": "string",
    "source_game_id": "string",
    "game_id": "string",
    "country_code": "string",
}

tables = {}

for csv_path in sorted(RELEASE_DIR.glob("*.csv")):
    tables[csv_path.stem] = pd.read_csv(
        csv_path,
        encoding="utf-8-sig",
        keep_default_na=False,
        na_values=[""],
        dtype=identifier_types,
    )

required_tables = {
    "tournaments",
    "placements",
    "twitch_monthly",
    "game_monthly_panel",
    "country_prizes",
    "country_audit",
}

missing_tables = required_tables - tables.keys()

if missing_tables:
    raise FileNotFoundError(
        f"Thiếu bảng cần kiểm tra: {sorted(missing_tables)}"
    )

# Đặt tên ngắn cho các bảng chính.
tournaments = tables["tournaments"]
placements = tables["placements"]
twitch = tables["twitch_monthly"]
panel = tables["game_monthly_panel"]
countries = tables["country_prizes"]
country_audit = tables["country_audit"]

# Chuẩn hóa các trường ngày của bảng chính trong bộ nhớ.
for table_name, date_columns in {
    "tournaments": ["start_date", "end_date", "month_start"],
    "twitch_monthly": ["month_start"],
    "game_monthly_panel": ["month_start"],
}.items():
    for column in date_columns:
        tables[table_name][column] = pd.to_datetime(
            tables[table_name][column],
            errors="raise",
        )

table_inventory = pd.DataFrame([
    {
        "Bảng": name,
        "Số dòng": len(frame),
        "Số cột": len(frame.columns),
    }
    for name, frame in tables.items()
])

print("Release:", RELEASE_DIR.name)
print("Số bảng CSV đã nạp:", len(tables))

with pd.option_context("display.max_rows", None):
    display(table_inventory)

Release: analysis_release_v1_20261005T194400349609Z
Số bảng CSV đã nạp: 36


,Bảng,Số dòng,Số cột
0,country_audit,14082,10
1,country_candidate_rows_review_only,61156,10
2,country_coverage,48,7
3,country_prizes,57597,9
4,country_quarantine,1534,10
5,country_year,3203,6
6,data_dictionary,158,5
7,earnings_out_of_scope,286,6
8,earnings_quarantine,1554,8
9,eda_annual_source_counts,48,3


In [20]:
strict_text = (
    tournaments["strict_quality"]
    .astype("string")
    .str.strip()    
    .str.lower()
)

print(
    "Số lượng strict_quality khác nhau:",
    strict_text.nunique(dropna=False),
    "Giá trị:",
    strict_text.unique(),
)

if not strict_text.isin(["true", "false"]).all():
    raise ValueError(
        "strict_quality có giá trị thiếu hoặc không phải True/False."
    )

strict_count = int(strict_text.eq("true").sum())

print(
    "Số lượng strict_quality = True:", strict_count
)

Số lượng strict_quality khác nhau: 2 Giá trị: <StringArray>
['true', 'false']
Length: 2, dtype: string
Số lượng strict_quality = True: 12540


In [23]:

strict_text = (
    tournaments["strict_quality"]
    .astype("string")
    .str.strip()
    .str.lower()
)

if not strict_text.isin(["true", "false"]).all():
    raise ValueError(
        "strict_quality có giá trị thiếu hoặc không phải True/False."
    )

strict_count = int(strict_text.eq("true").sum())

comparison_rows = []

def compare_count(metric, actual, expected, reference):
    actual = int(actual)
    expected = int(expected)

    comparison_rows.append({
        "Chỉ tiêu": metric,
        "Thực tế": actual,
        "Kỳ vọng": expected,
        "Chênh lệch": actual - expected,
        "Kết quả": "ĐẠT" if actual == expected else "KHÔNG ĐẠT",
        "Căn cứ": reference,
    })

compare_count(
    "Số giải đấu",
    len(tournaments),
    manifest_data["tournaments"],
    "manifest.tournaments",
)

compare_count(
    "Số placements",
    len(placements),
    manifest_data["placements"],
    "manifest.placements",
)

compare_count(
    "Số dòng Twitch chính",
    len(twitch),
    manifest_data["twitch_rows"],
    "manifest.twitch_rows",
)

compare_count(
    "Số giải strict_quality",
    strict_count,
    manifest_data["strict_tournaments"],
    "manifest.strict_tournaments",
)

compare_count(
    "Số dòng quốc gia",
    len(countries),
    manifest_data["country_rows"],
    "manifest.country_rows",
)

# Đối chiếu cả trạng thái quốc gia xuất hiện ngoài manifest, nếu có.
if country_audit["status"].isna().any():
    raise ValueError("country_audit có dòng thiếu status.")

actual_status_counts = (
    country_audit["status"].value_counts().to_dict()
)
expected_status_counts = manifest_data["country_statuses"]

for status in sorted(
    set(actual_status_counts) | set(expected_status_counts)
):
    compare_count(
        f"Country audit: {status}",
        actual_status_counts.get(status, 0),
        expected_status_counts.get(status, 0),
        f"manifest.country_statuses.{status}",
    )

# Manifest không có trường số dòng panel riêng.
# Theo thiết kế, panel phải có số dòng bằng chuỗi Twitch chính.
compare_count(
    "Số dòng panel",
    len(panel),
    len(twitch),
    "Quy tắc thiết kế: số dòng panel = số dòng Twitch",
)

# Country audit có một dòng cho mỗi giải.
compare_count(
    "Số dòng country audit",
    len(country_audit),
    len(tournaments),
    "Quy tắc thiết kế: một dòng audit cho mỗi giải",
)

scale_check = pd.DataFrame(comparison_rows)

display(scale_check)

failed_checks = scale_check[
    scale_check["Kết quả"].eq("KHÔNG ĐẠT")
]

if not failed_checks.empty:
    raise AssertionError(
        "Quy mô không khớp. Xem các dòng KHÔNG ĐẠT phía trên."
    )

print("Quy mô dữ liệu khớp manifest và các quy tắc số dòng đã kiểm tra.")
print(
    "Đã xác minh đầy đủ độ phủ nguồn:",
    manifest_data["final_source_coverage_verified"],
)

,Chỉ tiêu,Thực tế,Kỳ vọng,Chênh lệch,Kết quả,Căn cứ
0,Số giải đấu,14082,14082,0,ĐẠT,manifest.tournaments
1,Số placements,52551,52551,0,ĐẠT,manifest.placements
2,Số dòng Twitch chính,369,369,0,ĐẠT,manifest.twitch_rows
3,Số giải strict_quality,12540,12540,0,ĐẠT,manifest.strict_tournaments
4,Số dòng quốc gia,57597,57597,0,ĐẠT,manifest.country_rows
5,Country audit: reconciled,12548,12548,0,ĐẠT,manifest.country_statuses.reconciled
6,Country audit: total_mismatch,1076,1076,0,ĐẠT,manifest.country_statuses.total_mismatch
7,Country audit: unavailable,458,458,0,ĐẠT,manifest.country_statuses.unavailable
8,Số dòng panel,369,369,0,ĐẠT,Quy tắc thiết kế: số dòng panel = số dòng Twitch
9,Số dòng country audit,14082,14082,0,ĐẠT,Quy tắc thiết kế: một dòng audit cho mỗi giải


Quy mô dữ liệu khớp manifest và các quy tắc số dòng đã kiểm tra.
Đã xác minh đầy đủ độ phủ nguồn: False
